# Week 14 — Wednesday: Interpretability

**DATA 202 · Calvin University**

> "Magical technologies work … mostly. When they fail and you ask for an explanation, they present you with a blank face."
> — Fourcade & Healy, *The Ordinal Society*, ch. 2

A lender that denies a loan must tell the applicant the **principal reasons** (the Equal Credit Opportunity Act). In 2022, the Consumer Financial Protection Bureau added that a complex algorithm is no excuse: the reasons must still be specific and accurate.

Monday's model was a line, with slopes anyone can read. Today: a random forest, more accurate, with no slopes at all.

> 🎯 **Can a forest say why it denied someone?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Two denials (SLO 14B) |
| ~8 min | 2 · Which columns matter? (SLO 14B) |
| ~8 min | 3 · How does a column matter? (SLO 14B) |
| ~12 min | 4 · Why this applicant? (SLO 14B) |
| ~7 min | 5 · A toolbox for interpretability: which method, when? (SLO 14B) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: Home Mortgage Disclosure Act (HMDA) records for 2025, Consumer Financial Protection Bureau (public domain): applications for the applicant's own single-family home in Kent County, Michigan</sub>

---
## 1 · Two Denials (SLO 14B) · ~5 min

Week 11's applications and split. The forest gets Week 11's six columns, plus the applicant's yearly `income` and the home's `property_value`. A few applications leave those two blank: scikit-learn's trees accept blanks, and at each split the blanks go to the side that helps most.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

mortgages = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/mortgages.csv")
complete = mortgages.dropna(subset=["debt_to_income", "loan_to_value", "loan_amount"])      # Week 11's applications
numbers = ["debt_to_income", "loan_to_value", "loan_amount", "income", "property_value"]
words = ["loan_purpose", "lien", "loan_type"]

X = complete[numbers + words]
y = complete["decision"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)

forest = Pipeline([
    ("prep", ColumnTransformer([("words", OneHotEncoder(handle_unknown="ignore"), words)],
                               remainder="passthrough", verbose_feature_names_out=False)),     # numbers stay as they are
    ("forest", RandomForestClassifier(min_samples_leaf=5, random_state=1, n_jobs=-1)),
])
forest.fit(X_train, y_train)
print(f"forest: {forest.score(X_test, y_test):.3f} | Week 11's line: 0.882 | approve everyone: {(y_test == 'approved').mean():.3f}")

In [ ]:
applicants = complete.loc[[3389, 17464], numbers + words + ["decision", "denial_reason"]]     # two test applicants
applicants.index = ["A", "B"]
applicants["forest's chance of denial"] = forest.predict_proba(applicants[numbers + words])[:, 1].round(2)
applicants.T

Two test applicants, both denied by their lenders, both with a **0.74** chance of denial from the forest:

* **A** is buying a $205,000 home on $60,000 a year, borrowing 90% of its value. Debt-to-income: 50–60%. The lender's reason: **debt-to-income ratio**
* **B** wants a $35,000 home-improvement loan, a second lien, on a $305,000 home. Debt-to-income: 30–36%. With the mortgage B already has, the loans would add up to 101% of the home's value. The lender's reason: **collateral** (the home isn't worth enough to secure the loans)

Week 11's model had one slope per column: read the slopes and you see the reasons. This forest has 100 trees, each with hundreds of leaves.

> 🎯 **Predict First:** Which columns will the forest blame, for A and for B? (Part 4 answers.)

---
## 2 · Which Columns Matter? (SLO 14B) · ~8 min

A **global** question, about the forest as a whole: which columns does it rely on? Two ways to ask:

* **Impurity importance**, built into every forest: how much each column's splits cleaned up the leaves while the trees were learning
* **Permutation importance:** shuffle one column of the test set, so its values land on the wrong applicants, and measure how much accuracy drops. If the forest relies on a column, shuffling it hurts

> 📄 **Handout 1 — Two importances.** Circle the column the two disagree about most. 🗣️ What could the forest have been doing with it?

In [ ]:
impurity = pd.Series(forest[-1].feature_importances_, index=forest[0].get_feature_names_out())
for column in words:                                       # add up each word column's one-hot columns
    impurity[column] = impurity[impurity.index.str.startswith(column + "_")].sum()

shuffled_test = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=1)
shuffled_train = permutation_importance(forest, X_train, y_train, n_repeats=10, random_state=1)
importance = pd.DataFrame({"impurity (training)": impurity[numbers + words],
                           "permutation (test): accuracy lost": shuffled_test.importances_mean,
                           "permutation (training): accuracy lost": shuffled_train.importances_mean},
                          index=numbers + words).sort_values("impurity (training)")
bars = importance.iloc[:, :2].reset_index(names="column").melt(id_vars="column", var_name="importance", value_name="value")
fig = px.bar(bars, x="value", y="column", facet_col="importance", orientation="h", labels={"value": "", "column": ""},
             title="Two Importances for the Same Forest, Each on Its Own Scale")
fig.update_xaxes(matches=None)                             # don't share the x-axis between the two panels
fig.show()
importance.round(4)

<details><summary>Answer</summary>

* Both put **debt-to-income** first: 0.43 of the impurity, and shuffling it costs 0.092 of accuracy (90.5% → 81%)
* **Income** splits them. The forest split on it a lot while learning: 0.12 of the impurity, third place. But shuffling income in the test set costs almost nothing: **0.0001**
* On the training applications, shuffling income costs 0.010, a hundred times more. The forest's income splits fit the training applications, and they don't help on new ones: Week 7's **overfitting**, hiding inside an importance score
* Impurity importance comes from the training data, and it favors columns with many different values, like income and the property value. Prefer **permutation importance on held-out data**
* A catch for both: related columns share the credit. Loan-to-value is built from the loan amount and the property value
</details>

A second global tool: imitate the forest with a tree small enough to read, a **global surrogate**. The tree learns the *forest's* decisions, not the lenders':

In [ ]:
surrogate = Pipeline([
    ("prep", ColumnTransformer([("words", OneHotEncoder(handle_unknown="ignore"), words)],
                               remainder="passthrough", verbose_feature_names_out=False)),
    ("tree", DecisionTreeClassifier(max_depth=3, random_state=1)),
])
surrogate.fit(X_train, forest.predict(X_train))             # the forest's decisions are the labels
agree = (surrogate.predict(X_test) == forest.predict(X_test)).mean()
print(f"the tree agrees with the forest on {agree:.1%} of the test applications"
      f" | always 'approved' would agree on {(forest.predict(X_test) == 'approved').mean():.1%}")
print(export_text(surrogate[-1], feature_names=list(surrogate[0].get_feature_names_out())))

<details><summary>Answer</summary>

* Depth 3 agrees with the forest on **97%** of the test applications, against 92% for "always approved"
* Two rules carry most of it: a debt-to-income **above 60%** → denied (VA purchases aside); and a home worth **$90,000 or less**, unless it's a purchase → denied
* The catch: the surrogate explains an imitation. On the 3% where it disagrees with the forest, often the borderline cases, it says nothing
</details>

---
## 3 · How Does a Column Matter? (SLO 14B) · ~8 min

Importance says *how much*, not *how*. A **partial dependence plot (PDP)** shows how: set every applicant's debt-to-income to 15%, ask the forest, and average the chances. Then 25%, then 33%, and so on. Before averaging, each applicant has their own line: an **ICE curve** (individual conditional expectation).

> 📄 **Handout 2 — One line per applicant.** Label the cliffs. Circle a line that doesn't follow the average. 🗣️ The average says 50–60% means a chance of about one half. Whose chance is that?

In [ ]:
dti_values = [15, 25, 33] + list(range(36, 50)) + [55, 65]      # HMDA's values: 15, 25, 33, 55 and 65 stand for ranges
sample = X_test.sample(100, random_state=1)                       # 100 test applicants
ice = pd.DataFrame({dti: forest.predict_proba(sample.assign(debt_to_income=dti))[:, 1] for dti in dti_values},
                   index=sample.index)

lines = ice.reset_index(names="applicant").melt(id_vars="applicant", var_name="debt_to_income", value_name="chance")
fig = px.line(lines, x="debt_to_income", y="chance", line_group="applicant",
              labels={"debt_to_income": "debt-to-income (%), set for every applicant", "chance": "forest's chance of denial"},
              title="Debt-to-Income: One Line per Applicant (ICE) and Their Average (Partial Dependence)")
fig.update_traces(line_color="#b0b0b0", line_width=1)
fig.add_scatter(x=dti_values, y=ice.mean(), mode="lines+markers", name="average: partial dependence",
                line=dict(color="#1f4e99", width=4))
fig.show()
print("lines that end below 0.5:", (ice[65] < 0.5).sum(), "| lines that start above 0.5:", (ice[15] > 0.5).sum())

<details><summary>Answer</summary>

* **The average** is flat, about 0.13, up to 49%. Then it climbs: 0.50 at 50–60% and 0.84 above 60%. Week 11's staircase: a forest follows the cliffs that one S-curve couldn't
* **The lines** mostly jump at 50% and again at 60%, but by different amounts: at 50%, from nothing to +0.78. Three start high and stay high: one borrows 166% of the home's value, two have no home value on record. One stays low even above 60%: a VA home purchase, the surrogate's exception
* "About one half at 50–60%" is nobody's chance: it averages applicants for whom debt-to-income decides everything with applicants for whom it decides little. When ICE lines spread or cross, the column **interacts** with others, and the PDP alone hides it
* A catch: setting one column for everyone makes applicants who don't exist. Set loan-to-value to 100% while keeping the loan and the home's value, and the numbers contradict each other. **ALE plots** (accumulated local effects) use only small, realistic changes
</details>

---
## 4 · Why This Applicant? (SLO 14B) · ~12 min

Global tools describe the forest on average. A and B each want *their own* reason: a **local** explanation.

First, what-ifs. Change A's debt-to-income and loan-to-value, keep everything else, and ask the forest each time. Then the same for B.

> 📄 **Handout 3 — Around A and B.** Shade the cells where the forest denies (0.5 or more). Circle each applicant's own cell, and draw an arrow to the nearest cell where the forest approves. 🗣️ What would you write to A, and to B?

In [ ]:
grids = {}
for name in ["A", "B"]:
    what_if = pd.DataFrame([{"debt_to_income": dti, "loan_to_value": ltv}
                            for dti in [33, 40, 45, 49, 55, 65] for ltv in [60, 70, 80, 90, 95, 101]])
    changed = applicants.loc[[name] * len(what_if), numbers + words].reset_index(drop=True)
    changed[["debt_to_income", "loan_to_value"]] = what_if                     # everything else stays the applicant's
    what_if["chance"] = forest.predict_proba(changed)[:, 1]
    grids[name] = what_if.pivot(index="debt_to_income", columns="loan_to_value", values="chance")
pd.concat(grids, names=["applicant", "debt_to_income"]).round(2)

<details><summary>Answer</summary>

* **A:** every cell with a debt-to-income of 49% or less is approved (chances under 0.1), whatever the loan-to-value. A's reason is the debt-to-income
* **B:** at B's own debt-to-income, every loan-to-value of 90% or less is approved. B's reason is the loan-to-value: the lender's "collateral"
* Both match the reasons the lenders gave
* "If your debt-to-income were under 50%, you would be approved" is a **counterfactual explanation**: the smallest change that flips the decision. It's the explanation an applicant can act on
* Not every counterfactual is fair to give. "If you lived in another neighborhood" or "if you were younger" names something a person can't, or shouldn't have to, change
</details>

The grid tried two columns. **LIME** (local interpretable model-agnostic explanations) asks about all of them at once:

1. Make 2,000 made-up applicants close to A: nudge each number a little, at random
2. Ask the forest for each one's chance of denial
3. Give the closest made-up applicants the most weight
4. Fit a weighted straight line (Week 10). Its slopes are the local reasons

In [ ]:
rng = np.random.default_rng(1)
local_slopes = {}
for name in ["A", "B"]:
    nearby = applicants.loc[[name] * 2000, numbers + words].reset_index(drop=True)
    nudge = pd.DataFrame(rng.normal(0, 1, size=(2000, 5)), columns=numbers)       # how far each number moves
    nearby["debt_to_income"] += 5 * nudge["debt_to_income"]                         # one nudge = 5 points
    nearby["loan_to_value"] += 5 * nudge["loan_to_value"]
    for column in ["loan_amount", "income", "property_value"]:
        nearby[column] *= np.exp(0.1 * nudge[column])                               # one nudge = about 10%
    chance = forest.predict_proba(nearby)[:, 1]
    closeness = np.exp(-(nudge ** 2).sum(axis=1) / 4)                               # the closest count the most
    line = LinearRegression().fit(nudge, chance, sample_weight=closeness)
    local_slopes[name] = line.coef_                                                 # change in chance per nudge
pd.DataFrame(local_slopes, index=numbers).round(3)

<details><summary>Answer</summary>

* **A:** debt-to-income moves the chance by about **0.3** per nudge (5 points); every other column, by almost nothing
* **B:** loan-to-value leads, debt-to-income barely matters
* The same forest gives two different reasons, and each matches the lender's. Globally, debt-to-income came first (Part 2). For B, it isn't the reason: **local and global answers differ**
* A catch: LIME's answer depends on choices, like the size of a nudge and how much weight the closest get. Change them, and the slopes change. Before trusting an explanation, try another seed or nudge size
</details>

Herman Dooyeweerd, a Reformed philosopher, described fifteen **aspects** of reality, from the numerical to the ethical. Each answers "why?" in its own way. Why was A denied?

| aspect | an answer |
|---|---|
| numerical | the forest's chance was 0.74, above 0.5 |
| analytical | debt-to-income pushed the chance the most (LIME) |
| lingual | the notice says "debt-to-income ratio" |
| economic | the payments would take more than half of A's income |
| juridical | the law requires specific, accurate reasons, and forbids reasons like race |
| ethical | ? |

> 💬 **Question:** Our tools answer in the numerical and analytical aspects. What would a good answer in the ethical aspect look like, for A?

<details><summary>Answer</summary>

* One that serves A: a reason A can act on (the counterfactual: get the debt-to-income under 50%), and a path to it: how much debt to pay down, or a smaller home, and who could help
* Honest about its limits: the model never saw A's credit history, and an explanation can only name columns the model has
* An explanation is an answer to someone's "why". A good one is shaped by care for the person asking, not only by the model
</details>

---
## 5 · A Toolbox for Interpretability: Which Method, When? (SLO 14B) · ~7 min

**The toolbox:** every method from today and from the old notes, the question it answers, when to reach for it, and how to call it. **Global** = the model as a whole; **local** = one prediction. **Intrinsic** = the model explains itself; **post-hoc** = explained from outside, for any model.

| | method | the question it answers | global or local | when to use it | how, in code |
|---|---|---|---|---|---|
| | ***Read the model itself (intrinsic)*** | | | | |
| 1 | **Slopes** | how much does each column push the score? | global; local as slope × value | when a linear or logistic model is accurate enough. Scale first; related columns can flip signs (Week 10) | `model[-1].coef_` |
| 2 | **Tree rules** | which conditions lead to each decision? | global; local as one path | when a small tree is accurate enough | `export_text(tree, feature_names=...)` |
| | ***Explain any model from outside (post-hoc)*** | | | | |
| 3 | **Impurity importance** | which columns did the splits use? | global | a quick first look at a forest. Training data only: it can reward overfitting (income) | `forest.feature_importances_` |
| 4 | **Permutation importance** | how much accuracy is lost when a column is shuffled? | global | which columns the model relies on, on new data. Related columns share the credit | `permutation_importance(model, X_test, y_test)` |
| 5 | **Global surrogate** | which simple rules imitate the model? | global | a readable summary. Report how often it agrees | `DecisionTreeClassifier(max_depth=3).fit(X, model.predict(X))` |
| 6 | **Partial dependence (PDP)** | how does the average prediction change with one column? | global | the shape of an effect: cliffs, curves. Averages hide interactions; related columns make impossible rows | `PartialDependenceDisplay.from_estimator(model, X, ["column"])` |
| 7 | **ALE** | the same, with only small, realistic changes | global | when the column is related to others | `PyALE` or `alibi` packages |
| 8 | **ICE** | how does each applicant's prediction change with one column? | local, many drawn at once | finding interactions; one person's what-ifs | `kind="individual"` or `kind="both"` in 6 |
| 9 | **Counterfactual** | what is the smallest change that flips the decision? | local | giving a reason a person can act on. Many answers exist; some changes aren't possible, or fair to ask | a what-if grid (Part 4); `dice-ml` package |
| 10 | **LIME** | which columns push this prediction, nearby? | local | explaining one decision of any model. Depends on the nudges and weights: check it's stable | `lime` package: `LimeTabularExplainer(X_train.values, ...).explain_instance(row, model.predict_proba)` |
| 11 | **SHAP** | how much of this prediction, against the average, is each column's fair share? | local; global when averaged | one decision, with shares that add up exactly (Shapley values, from game theory) | `shap` package: `shap.TreeExplainer(forest)`, or `shap.Explainer(model.predict_proba, X_train)` |

**More:** *prototypes and criticisms* (the applicants nearest the k-means centers of Week 5 typify the data, and the ones far from every center don't fit it); the *H-statistic* measures how much two columns interact (slow).

**How to choose:**

* **First ask whether you need a black box.** If a readable model (1–2) is nearly as accurate, use it. For high-stakes decisions, an explanation of a black box is a second model that can be wrong (Rudin, 2019). Here: forest 90.5%, Week 11's line 88.2%
* **Global questions** (which columns, what shape, what rules?): 4, then 6 with 8, then 5
* **Local questions** (why this person?): 10 or 11 for what pushed the prediction; 9 for what would change it
* **Always** on held-out data, and with more than one method. Explanations can also audit: run them on Monday's model with the neighborhood, and the proxy shows up
* Some models need little explaining: low stakes (a song recommendation), or when explaining would help people game the system (fraud detection)

> 📄 **Handout 4 — Pick the method.** Write the letter that fits each situation best. 🗣️ Could an explanation hide a proxy?

<details><summary>Answer</summary>

1. **E.** A counterfactual: what to change to be approved
2. **A.** Permutation importance, on new patients (held-out data)
3. **B.** Partial dependence, with ICE lines to check that the average holds for most days
4. **C.** A global surrogate, with how often it agrees with the 500 trees
5. **D.** LIME or SHAP: one patient's prediction
6. **F.** One point of accuracy for a model the board can read, in a decision about someone's freedom

* **A hidden proxy:** an explanation names the columns a model uses, not what those columns stand for. "Loan-to-value" looks neutral; Monday's "neighborhood" did too. To check, audit the decisions by group (Monday) and test whether the explaining columns predict race
</details>

---
### Wrap-up

1. A model is **interpretable** by design (slopes, small trees), or **explained** from outside (post-hoc)
2. **Global** tools describe the model (importance, partial dependence, surrogates); **local** tools explain one decision (ICE, LIME, SHAP, counterfactuals). The two can disagree
3. Prefer **permutation importance on held-out data**: impurity importance can reward overfitting 📌
4. **Averages hide interactions:** look at the ICE lines under a partial dependence plot
5. The explanation an applicant can use is a **counterfactual**, and the law asks for specific, accurate reasons
6. Part 5's table is the checklist for explaining any model

**Friday:** Quiz 6.